<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="./img/btp-banner.gif" alt="BTP A&C">
</div>

**SAP-RPT-1 in Action** — Architecting Agentic Supply Chain on SAP Business AI Platform

# Exercise 1B — Can You Trust the Prediction?
## Use Case: Just-In-Time (JIT) Supply Chain Risk — the Evaluation Lens

In Exercise 1A you put SAP-RPT-1 to work: you predicted a PO's delay and turned it into a Green/Amber/Red action. This exercise asks the question every architect must answer *before* letting that prediction drive a real decision:

> **How do I know the model is good enough — and which number proves it?**

### Recap: the use case (same as 1A)

**BestRun Technologies** is a fabless maker of IoT security devices in a **cornered JIT position** — sole-sourced critical components, no viable safety-stock buffer, and a booked production line that costs **$15,000+/hour** when a critical shipment slips. Early warning is the only lever, so SAP-RPT-1's delay prediction is the primary defense. That makes the *quality* of the prediction a business-critical question, not an academic one.

### Where this sits on the Decision Ladder

Exercise 1B **stays parked on the Predict rung** — it adds no new capability. It does the thing a responsible architect does before operationalizing a model: **measure it against a baseline, with metrics chosen from the business decision.**

| Rung | | Exercise |
|------|--|----------|
| 🔵 Rules | Threshold policy | 1A, Section 3 |
| 🟢 **Predict** | **SAP-RPT-1 — and *is it good enough?*** | **1A + 1B (you are here)** |
| 🟠 Reason | ReAct agent | Exercise 2 |

### How this exercise is structured

Same rhythm as 1A — short sections, each one proves something:

> 📘 **KNOWLEDGE POINT** — the idea &nbsp;→&nbsp; 🎯 **Outcome** — what you'll be able to do &nbsp;→&nbsp; ⌨️ **Your Turn** — you run it &nbsp;→&nbsp; ✓ **Checkpoint** — what you proved

| # | Section | You'll prove |
|---|---------|--------------|
| **1** | Setup & Data | Live connection + the evaluation dataset |
| **2** | Frame the Question Two Ways | *Severity* and *routing* are different questions |
| **3** | Open-Source Regression Baselines | A model is only "good" relative to a baseline |
| **4** | SAP-RPT-1 Holdout Evaluation | How SAP-RPT-1 scores on the same holdout |
| **5** | Intervention-Risk Classification | Precision vs recall *is* a business choice |
| **6** | Score New Scenarios | Whether the models drive the same **action** |
| **7** | **Apply to YOUR Landscape** | Pick *your* metric from *your* decision |
| **8** | → From Workshop to Production | The questions that come next |

**You are here → Section 0: Orientation.**


---

## Section 1 — Setup & Data

📘 **KNOWLEDGE POINT — Evaluation needs the same discipline as production**
A benchmark you can't trust is worse than no benchmark — it manufactures false confidence. So we fail fast on credentials, pin our dependency versions, and load a dataset that is **kept separate from Exercise 1A** so the two exercises never contaminate each other.

🎯 **Outcome:** a validated live connection to SAP-RPT-1 and the 1B evaluation dataset loaded, with its risk-tier balance visible.


### Step 1.1 — Install packages

Same stack as 1A, plus **scikit-learn** — the open-source baselines we'll benchmark SAP-RPT-1 against.


In [ ]:
%pip install generative-ai-hub-sdk==4.12.4 --quiet
%pip install pandas python-dotenv requests scikit-learn --quiet

print("Packages installed successfully. Restart the kernel if prompted.")

### Step 1.2 — Imports, config, and fail-fast credential check

All imports, the **evaluation constants** (risk thresholds, the 80/20 holdout split, a fixed `RANDOM_STATE=42` so results are reproducible), and the same fail-fast guard from 1A: if any credential is missing, stop now with a clear message rather than deep inside an API call.


In [ ]:
import math
import os
import time
from typing import Dict, List, Tuple

import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, precision_score, r2_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

load_dotenv(find_dotenv(), override=False)

RISK_THRESHOLD_AMBER = 1.0
RISK_THRESHOLD_RED = 3.0
TEST_SIZE = 0.20
RANDOM_STATE = 42
TOKEN_TIMEOUT_SECONDS = 30
HTTP_TIMEOUT_SECONDS = 60
TOKEN_REFRESH_BUFFER_SECONDS = 60
RPT1_MAX_RETRIES = 5
RPT1_RETRY_BACKOFF_SECONDS = 2.0
RPT1_MIN_REQUEST_INTERVAL_SECONDS = 0.35

AICORE_AUTH_URL = os.getenv("AICORE_AUTH_URL")
AICORE_CLIENT_ID = os.getenv("AICORE_CLIENT_ID")
AICORE_CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET")
AICORE_BASE_URL = os.getenv("AICORE_BASE_URL")
AICORE_RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP")
RPT1_DEPLOYMENT_URL = os.getenv("RPT1_DEPLOYMENT_URL")

required = {
    "AICORE_AUTH_URL": AICORE_AUTH_URL,
    "AICORE_CLIENT_ID": AICORE_CLIENT_ID,
    "AICORE_CLIENT_SECRET": AICORE_CLIENT_SECRET,
    "AICORE_BASE_URL": AICORE_BASE_URL,
    "AICORE_RESOURCE_GROUP": AICORE_RESOURCE_GROUP,
    "RPT1_DEPLOYMENT_URL": RPT1_DEPLOYMENT_URL,
}

missing = [k for k, v in required.items() if not v]
if missing:
    raise EnvironmentError(
        "Missing required environment variables: "
        f"{', '.join(missing)}.\n"
        "This exercise benchmarks the live SAP-RPT-1 model on SAP AI Core and "
        "cannot run without valid credentials.\n"
        "Fix: create a .env file in the project root (copy .env.example), fill in "
        "every key listed above, then restart the kernel and re-run this cell."
    )

print("Configuration valid. Ready to benchmark against live SAP-RPT-1.")


### Step 1.3 — Load the evaluation dataset

These files (`*_part1b.csv`) are **isolated from Exercise 1A** so the original workshop data stays untouched. We derive the same Green/Amber/Red tiers and an `Needs_Intervention` flag (delay past the Red threshold) — this is the *ground truth* every metric below is scored against.

⌨️ **Your Turn:** run the cell and read the **risk-tier distribution**. Note how many POs are truly Red — that class balance is what makes recall (Section 5) a real challenge.


In [ ]:
historical_df = pd.read_csv("data/historical_po_data_part1b.csv")
prediction_df = pd.read_csv("data/new_po_prediction_part1b.csv")

def derive_risk_tier(delay_days: float) -> str:
    if delay_days < RISK_THRESHOLD_AMBER:
        return "Green"
    if delay_days <= RISK_THRESHOLD_RED:
        return "Amber"
    return "Red"

historical_df["Risk_Tier"] = historical_df["Actual_Delay_Days"].apply(derive_risk_tier)
historical_df["Needs_Intervention"] = historical_df["Actual_Delay_Days"] > RISK_THRESHOLD_RED

print(f"Historical rows: {len(historical_df):,}")
print(f"Prediction scenarios: {len(prediction_df):,}")
print("Risk tier distribution:")
display(historical_df["Risk_Tier"].value_counts().rename_axis("Risk_Tier").to_frame("Count"))

> ✓ **CHECKPOINT — Section 1**
> You have a live SAP-RPT-1 connection and the evaluation dataset loaded, and you've seen how many orders are actually Red.
>
> 🏭 **What would break in production?** This dataset is a fixed snapshot. In production, supplier behavior drifts — a vendor that was reliable last quarter degrades. *How would you keep the holdout honest and re-validate over time?* (Held for the Architecture Playbook.)


---

## Section 2 — Frame the Question Two Ways

📘 **KNOWLEDGE POINT — The business decision picks the model formulation, not the other way around**
The same PO data can answer two *different* questions, and they serve different decisions:

| Frame | Question | Best for the decision... | Metric family |
|-------|----------|--------------------------|---------------|
| **Regression** | *How many days late?* | Sizing severity & business impact | MAE / RMSE / R² |
| **Classification** | *Will it cross the Red line and need intervention?* | Alert routing, planner attention, escalation | Precision / Recall / F1 |

An architect who reaches for a metric before naming the decision is optimizing the wrong thing. We build **both** frames on the *same features and the same holdout split* so the comparison is fair.

🎯 **Outcome:** one train/test split, reused by every model in the notebook — the foundation of a fair benchmark.


### Step 2.1 — Prepare features and the shared holdout split

We keep the same structured ERP features from 1A. One `train_test_split` — **stratified on the intervention flag** so both splits keep a representative share of rare Red cases — feeds every model below. Reusing one split is what makes the numbers comparable.

> *Workshop-grade note:* we use the provided supplier fields as-is. A production benchmark would recompute derived supplier features on **train data only**, to avoid leaking future information into the test set.


In [ ]:
feature_columns = [
    "Vendor_ID", "Vendor_Country", "Vendor_OTIF_Percent", "Vendor_Avg_Past_Delay",
    "Material_ID", "Material_Group", "Criticality_Flag", "Plant_ID",
    "Order_Quantity", "Net_Price", "Planned_Lead_Time_Days", "Order_Month", "Incoterms"
]
categorical_features = [
    "Vendor_ID", "Vendor_Country", "Material_ID", "Material_Group",
    "Criticality_Flag", "Plant_ID", "Incoterms"
]
numeric_features = [column for column in feature_columns if column not in categorical_features]

X = historical_df[feature_columns].copy()
y_reg = historical_df["Actual_Delay_Days"].copy()
y_cls = historical_df["Needs_Intervention"].astype(int).copy()

X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_reg, y_cls,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_cls
)

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features),
    ]
)

print(f"Train rows: {len(X_train):,} | Test rows: {len(X_test):,}")
print(f"Positive intervention cases in test split: {int(y_cls_test.sum())}")

> ✓ **CHECKPOINT — Section 2**
> You've framed the problem two ways and built the single stratified holdout that every model shares. You saw how few positive intervention cases land in the test split — that scarcity is exactly why recall matters later.
>
> 🏭 **What would break in production?** Stratifying on a rare class in a small sample is fragile. *How large a holdout do you need before a recall number is trustworthy?*


---

## Section 3 — Open-Source Regression Baselines

📘 **KNOWLEDGE POINT — "Good" is meaningless without a baseline**
A MAE of "1.2 days" tells you nothing on its own. Is that good? The only way to know is to compare it against a reference. So before we score SAP-RPT-1, we build two honest open-source baselines — a **linear model** and a **random-forest ensemble** — trained on the same split. SAP-RPT-1 has to *beat something* to earn trust.

🎯 **Outcome:** MAE / RMSE / R² for two baseline regressors — the bar SAP-RPT-1 must clear.


### Step 3.1 — Train and score the regression baselines

⌨️ **Your Turn:** run the cell. Read MAE first (average days off), then RMSE (does the model make *large* misses?). A model can have a decent MAE but a bad RMSE — that means it's usually fine but occasionally very wrong, which is dangerous for high-value Red orders.


In [ ]:
def evaluate_regression(y_true: pd.Series, y_pred: List[float]) -> Dict[str, float]:
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": math.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

regression_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        min_samples_leaf=2
    ),
}

regression_results = []
fitted_regressors = {}

for model_name, model in regression_models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    pipeline.fit(X_train, y_reg_train)
    predictions = pipeline.predict(X_test)
    metrics = evaluate_regression(y_reg_test, predictions)
    metrics["Model"] = model_name
    regression_results.append(metrics)
    fitted_regressors[model_name] = pipeline

regression_results_df = pd.DataFrame(regression_results).sort_values(["MAE", "RMSE"]).reset_index(drop=True)
display(regression_results_df)

> ✓ **CHECKPOINT — Section 3**
> You have a baseline. Any model — including SAP-RPT-1 — is now judged *relative to these numbers*, not in the abstract.
>
> 🏭 **What would break in production?** These baselines were trained here, in the notebook. *Who retrains them, how often, and how do you detect when the baseline itself has gone stale?*


---

## Section 4 — SAP-RPT-1 Holdout Evaluation

📘 **KNOWLEDGE POINT — Evaluate with batch inference, not a loop**
To score SAP-RPT-1 on the holdout we could call the API once per test row — slow, fragile, and rate-limit-prone. The architecturally correct pattern is **batch inference**: pack *all* test rows into one request, each carrying `[PREDICT]` in the target column alongside a sample of context rows. One call, one response, predictions in order. This is the same `[PREDICT]` contract from 1A, applied at scale.

🎯 **Outcome:** SAP-RPT-1's MAE / RMSE / R² on the *same* holdout, dropped into the same table as the baselines.


### Step 4.1 — Batch-score SAP-RPT-1 and compare

This cell defines the `AICoreClient` (token caching, retry/backoff on 401/429 — same robust client as 1A), the batch and single-row prediction functions, then runs the batch holdout evaluation and appends SAP-RPT-1 to the regression comparison.

⌨️ **Your Turn:** run it and read the combined table. **Where does SAP-RPT-1 land** relative to your baselines on MAE and RMSE? A tabular foundation model earns its keep when it matches or beats a tuned ensemble **with no training job** — that operational simplicity is part of its architectural value.


In [ ]:
class AICoreClient:
    def __init__(self, auth_url: str, client_id: str, client_secret: str, base_url: str, resource_group: str):
        self._auth_url = auth_url
        self._client_id = client_id
        self._client_secret = client_secret
        self._base_url = base_url
        self._resource_group = resource_group
        self._access_token: str | None = None
        self._token_expires_at: float = 0.0

    def _get_token(self) -> str:
        now = time.time()
        if self._access_token and now < (self._token_expires_at - TOKEN_REFRESH_BUFFER_SECONDS):
            return self._access_token

        response = requests.post(
            f"{self._auth_url}/oauth/token",
            data={"grant_type": "client_credentials"},
            auth=(self._client_id, self._client_secret),
            timeout=TOKEN_TIMEOUT_SECONDS,
        )
        response.raise_for_status()
        payload = response.json()
        self._access_token = payload["access_token"]
        self._token_expires_at = now + int(payload.get("expires_in", 600))
        return self._access_token

    def predict(self, deployment_url: str, payload: dict) -> dict:
        last_exception: Exception | None = None

        for attempt in range(1, RPT1_MAX_RETRIES + 1):
            token = self._get_token()

            try:
                response = requests.post(
                    deployment_url,
                    json=payload,
                    headers={
                        "Authorization": f"Bearer {token}",
                        "AI-Resource-Group": self._resource_group,
                        "Content-Type": "application/json",
                    },
                    timeout=HTTP_TIMEOUT_SECONDS,
                )
                response.raise_for_status()
                return response.json()
            except requests.HTTPError as exc:
                last_exception = exc
                resp = exc.response
                status_code = resp.status_code if resp is not None else None

                if status_code == 401 and attempt < RPT1_MAX_RETRIES:
                    self._access_token = None
                    time.sleep(RPT1_RETRY_BACKOFF_SECONDS * (2 ** (attempt - 1)))
                    continue

                if status_code == 429 and attempt < RPT1_MAX_RETRIES:
                    retry_after = resp.headers.get("Retry-After") if resp is not None else None
                    sleep_seconds = float(retry_after) if retry_after else RPT1_RETRY_BACKOFF_SECONDS * (2 ** (attempt - 1))
                    print(f"[INFO] SAP-RPT-1 rate limited; retrying in {sleep_seconds:.1f}s (attempt {attempt}/{RPT1_MAX_RETRIES})")
                    time.sleep(sleep_seconds)
                    continue

                raise
            except requests.RequestException as exc:
                last_exception = exc
                if attempt < RPT1_MAX_RETRIES:
                    sleep_seconds = RPT1_RETRY_BACKOFF_SECONDS * (2 ** (attempt - 1))
                    print(f"[INFO] SAP-RPT-1 request failed; retrying in {sleep_seconds:.1f}s (attempt {attempt}/{RPT1_MAX_RETRIES})")
                    time.sleep(sleep_seconds)
                    continue

                raise

        raise RuntimeError("SAP-RPT-1 inference failed after exhausting retries") from last_exception


aicore_client = AICoreClient(
    auth_url=AICORE_AUTH_URL,
    client_id=AICORE_CLIENT_ID,
    client_secret=AICORE_CLIENT_SECRET,
    base_url=AICORE_BASE_URL,
    resource_group=AICORE_RESOURCE_GROUP,
)


def predict_delay_rpt1_batch(
    test_df: pd.DataFrame,
    context_df: pd.DataFrame,
    feature_cols: List[str],
) -> Tuple[List[float], str]:
    """
    Predict delay for multiple rows in a single SAP-RPT-1 API call.

    Packs all test rows as [PREDICT] rows alongside context rows.
    Returns one prediction per test row in the same order.
    """
    columns = feature_cols + ["Actual_Delay_Days"]
    context_sample = context_df[columns].sample(
        n=min(200, len(context_df)), random_state=RANDOM_STATE
    )

    # Build prediction rows — each gets [PREDICT] as the target
    pred_rows = []
    for _, row in test_df.iterrows():
        pred_dict = row[feature_cols].to_dict()
        pred_dict["Actual_Delay_Days"] = "[PREDICT]"
        pred_rows.append(pred_dict)

    all_rows = pd.concat(
        [context_sample, pd.DataFrame(pred_rows)], ignore_index=True
    ).to_dict("records")

    payload = {
        "rows": all_rows,
        "prediction_config": {
            "target_columns": [{
                "name": "Actual_Delay_Days",
                "prediction_placeholder": "[PREDICT]"
            }]
        }
    }

    deployment_url = RPT1_DEPLOYMENT_URL.rstrip("/")
    if not deployment_url.endswith("/predict"):
        deployment_url = f"{deployment_url}/predict"

    try:
        print(f"[INFO] Sending batch request: {len(context_sample)} context + {len(pred_rows)} predict rows")
        t0 = time.time()
        response = aicore_client.predict(deployment_url=deployment_url, payload=payload)
        elapsed = time.time() - t0
        print(f"[INFO] Batch response received in {elapsed:.1f}s")

        predictions_raw = response["predictions"]
        predictions = [
            float(p["Actual_Delay_Days"][0]["prediction"])
            for p in predictions_raw
        ]

        if len(predictions) != len(pred_rows):
            raise ValueError(
                f"Expected {len(pred_rows)} predictions, got {len(predictions)}"
            )

        return predictions, "sap-rpt-1"

    except Exception as exc:
        raise RuntimeError(
            "SAP-RPT-1 batch inference failed. "
            "Check that RPT1_DEPLOYMENT_URL points to an active deployment and that "
            "your AI Core credentials and resource group are correct."
        ) from exc


def predict_delay_rpt1(row: pd.Series, context_df: pd.DataFrame) -> Tuple[float, str]:
    """Single-row prediction (used by Step 6 for small scenario sets)."""
    columns = feature_columns + ["Actual_Delay_Days"]
    context_sample = context_df[columns].sample(n=min(200, len(context_df)), random_state=RANDOM_STATE)
    pred_row = row[feature_columns].to_dict()
    pred_row["Actual_Delay_Days"] = "[PREDICT]"
    rows = pd.concat([context_sample, pd.DataFrame([pred_row])], ignore_index=True).to_dict("records")
    payload = {
        "rows": rows,
        "prediction_config": {
            "target_columns": [{
                "name": "Actual_Delay_Days",
                "prediction_placeholder": "[PREDICT]"
            }]
        }
    }

    deployment_url = RPT1_DEPLOYMENT_URL.rstrip("/")
    if not deployment_url.endswith("/predict"):
        deployment_url = f"{deployment_url}/predict"

    try:
        response = aicore_client.predict(deployment_url=deployment_url, payload=payload)
        prediction_value = response["predictions"][0]["Actual_Delay_Days"][0]["prediction"]
        return float(prediction_value), "sap-rpt-1"
    except Exception as exc:
        raise RuntimeError(
            "SAP-RPT-1 inference failed after retries. "
            "Check that RPT1_DEPLOYMENT_URL points to an active deployment and that "
            "your AI Core credentials and resource group are correct."
        ) from exc


# --- Run batch holdout evaluation ---
rpt1_predictions, rpt1_method = predict_delay_rpt1_batch(
    test_df=X_test.reset_index(drop=True),
    context_df=historical_df,
    feature_cols=feature_columns,
)

rpt1_metrics = evaluate_regression(y_reg_test.reset_index(drop=True), rpt1_predictions)
rpt1_metrics["Model"] = "SAP-RPT-1"
rpt1_metrics["Inference_Mode"] = rpt1_method

regression_comparison_df = pd.concat([
    regression_results_df,
    pd.DataFrame([rpt1_metrics])
], ignore_index=True, sort=False).sort_values(["MAE", "RMSE"]).reset_index(drop=True)

display(regression_comparison_df)


> ✓ **CHECKPOINT — Section 4**
> You've benchmarked the live model against open-source baselines on identical data. You can now say — with a number — whether SAP-RPT-1 is good enough for *severity* estimation.
>
> 🏭 **What would break in production?** You batched 200 context rows here. *At production volume, what's your context-sampling strategy, your latency budget, and your fallback when the endpoint is slow or down?*


---

## Section 5 — Intervention-Risk Classification

📘 **KNOWLEDGE POINT — Precision vs recall is a business decision, not a math one**
For operational routing, planners often don't need the exact delay — they need to know: *does this PO cross the Red line and need a human?* That's classification, and its two errors cost different amounts:

- **Missing a true Red case** (low recall) → a line-down surprise: $15,000+/hour.
- **A false alarm** (low precision) → wasted planner attention, alert fatigue.

Which one you optimize is **the business's call**, not the data scientist's. We benchmark two open-source classifiers *and* SAP-RPT-1 thresholded at the Red line, so you can put the trade-off on the table.

🎯 **Outcome:** Precision / Recall / F1 + confusion matrices — the evidence for a routing-policy conversation.


### Step 5.1 — Benchmark the classifiers

⌨️ **Your Turn:** run it, then read the confusion matrices. For each model ask: *would BestRun rather tolerate this many false alarms, or this many missed Red orders?* There is no universally "best" row here — only the row that fits the business's tolerance for each error.


In [ ]:
def evaluate_classification(y_true: pd.Series, y_pred: List[int]) -> Dict[str, float]:
    return {
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
    }

classification_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="liblinear"
    ),
    "Random Forest Classifier": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        min_samples_leaf=2,
        class_weight="balanced"
    ),
}

classification_results = []
fitted_classifiers = {}

for model_name, model in classification_models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    pipeline.fit(X_train, y_cls_train)
    predictions = pipeline.predict(X_test)
    metrics = evaluate_classification(y_cls_test, predictions)
    metrics["Model"] = model_name
    metrics["Confusion_Matrix"] = str(confusion_matrix(y_cls_test, predictions).tolist())
    classification_results.append(metrics)
    fitted_classifiers[model_name] = pipeline

rpt1_class_predictions = [int(prediction > RISK_THRESHOLD_RED) for prediction in rpt1_predictions]
rpt1_cls_metrics = evaluate_classification(y_cls_test, rpt1_class_predictions)
rpt1_cls_metrics["Model"] = "SAP-RPT-1 Thresholded"
rpt1_cls_metrics["Confusion_Matrix"] = str(confusion_matrix(y_cls_test, rpt1_class_predictions).tolist())

classification_results_df = pd.concat([
    pd.DataFrame(classification_results),
    pd.DataFrame([rpt1_cls_metrics])
], ignore_index=True).sort_values(["F1", "Recall"], ascending=False).reset_index(drop=True)

display(classification_results_df)

> ✓ **CHECKPOINT — Section 5**
> You framed the same prediction as a routing decision and saw the precision/recall trade-off explicitly. You can now recommend a metric *because of* the business's cost of each error, not by default.
>
> 🏭 **What would break in production?** The Red threshold (3 days) is a policy knob. *Who owns it, and how does moving it re-shape every recall number downstream?*


---

## Section 6 — Score New Scenarios: Do the Models *Act* the Same?

📘 **KNOWLEDGE POINT — What matters operationally is the action, not the decimal**
Two models can predict 2.8 and 3.2 days for the same PO and disagree on the *number* — but if both land in the same risk tier, they drive the **same business action**. Conversely, small numeric gaps that straddle a threshold flip the action. When you compare models for deployment, compare the **decisions they produce**, not just their error metrics.

🎯 **Outcome:** the strongest baseline and SAP-RPT-1, side by side on fresh POs, compared by the *tier and intervention flag* they produce.


### Step 6.1 — Compare operational outputs

⌨️ **Your Turn:** run it. For each PO, do the open-source and SAP-RPT-1 paths assign the **same tier**? Where they agree, you have directional confidence. Where they differ, note whether the disagreement actually changes what a planner would *do* — that's the only difference that costs money.


In [ ]:
best_regressor_name = regression_results_df.sort_values(["MAE", "RMSE"]).iloc[0]["Model"]
best_classifier_name = classification_results_df.sort_values(["F1", "Recall"], ascending=False).iloc[0]["Model"]

best_regressor = fitted_regressors[best_regressor_name]
best_classifier = fitted_classifiers.get(best_classifier_name)
if best_classifier is None:
    best_classifier_name = "Random Forest Classifier"
    best_classifier = fitted_classifiers[best_classifier_name]

scenario_rows = prediction_df[feature_columns].copy()
prediction_df["OpenSource_Predicted_Delay"] = best_regressor.predict(scenario_rows).round(2)
prediction_df["OpenSource_Risk_Tier"] = prediction_df["OpenSource_Predicted_Delay"].apply(derive_risk_tier)
prediction_df["OpenSource_Intervention_Flag"] = best_classifier.predict(scenario_rows).astype(int)

rpt1_scores = []
rpt1_modes = []
for _, row in prediction_df.iterrows():
    score, mode = predict_delay_rpt1(row, historical_df)
    rpt1_scores.append(round(score, 2))
    rpt1_modes.append(mode)

prediction_df["SAP_RPT1_Predicted_Delay"] = rpt1_scores
prediction_df["SAP_RPT1_Risk_Tier"] = prediction_df["SAP_RPT1_Predicted_Delay"].apply(derive_risk_tier)
prediction_df["SAP_RPT1_Mode"] = rpt1_modes

summary_columns = [
    "PO_ID", "Vendor_ID", "Material_Group", "Criticality_Flag",
    "OpenSource_Predicted_Delay", "OpenSource_Risk_Tier", "OpenSource_Intervention_Flag",
    "SAP_RPT1_Predicted_Delay", "SAP_RPT1_Risk_Tier", "SAP_RPT1_Mode"
]

print(f"Best regression baseline: {best_regressor_name}")
print(f"Best classification baseline: {best_classifier_name}")
display(prediction_df[summary_columns])

> ✓ **CHECKPOINT — Section 6**
> You compared models by the *actions* they trigger, not just their metrics — the lens that actually matters for a deployment decision.
>
> 🏭 **What would break in production?** Model agreement today doesn't guarantee agreement after drift. *How would you monitor action-level agreement between a champion and a challenger model over time?*


---

## Section 7 — Apply to YOUR Landscape

📘 **KNOWLEDGE POINT — The transferable skill isn't the model, it's choosing the metric from the decision**
Everything in this notebook reduces to one reusable move: **name the decision first, then let it pick the metric.** Severity decisions → regression, lead with MAE (RMSE if large misses are dangerous). Act-or-not decisions → classification, and the *relative cost of the two errors* picks precision vs recall. Carry this to any Predict-rung use case, not just delay.

🎯 **Outcome:** you map one prediction decision from *your own* landscape to its evaluation frame and headline metric.

⌨️ **Your Turn:** edit the values in the cell below and run it. It's offline — no SAP call — so iterate freely.


In [ ]:
# =====================================================================
# ⌨️  YOUR TURN — map a prediction decision from YOUR landscape to a metric
#     Edit the five values below, then run. (Offline — no SAP call.)
# =====================================================================

MY_DECISION       = "Predict which invoices will be paid late"   # one line, in your words
DECISION_TYPE     = "routing"    # "severity" (how bad?) OR "routing" (act or not?)

# Only used for a "routing" decision — the relative cost of the two errors:
COST_OF_MISSED_CASE = "high"     # missing a true positive  -> "high" / "medium" / "low"
COST_OF_FALSE_ALARM = "medium"   # a false alarm            -> "high" / "medium" / "low"

# A quick reality check on whether Predict even applies:
HAS_HISTORICAL_LABEL = True      # do you have past outcomes to learn from / score against?


_RANK = {"low": 0, "medium": 1, "high": 2}

def recommend_evaluation(decision_type, missed_cost, false_alarm_cost, has_label):
    if not has_label:
        return ("No label, no benchmark",
                "Without historical outcomes you cannot score a model. Stay on the "
                "Rules rung, or invest in labelling before climbing to Predict.")

    if decision_type == "severity":
        return ("Regression",
                "Lead with MAE (average error, in the unit of the decision). "
                "Watch RMSE if occasional LARGE misses are dangerous — it penalises "
                "big errors harder. Use R2 only as a supporting 'explains-the-variance' check.")

    if decision_type == "routing":
        m, f = _RANK[missed_cost], _RANK[false_alarm_cost]
        if m > f:
            headline = ("Recall", "Missing a true case costs more than a false alarm, "
                        "so catch as many real cases as possible — accept some false alarms.")
        elif f > m:
            headline = ("Precision", "False alarms cost more (alert fatigue / wasted effort), "
                        "so only flag when you're confident — accept some misses.")
        else:
            headline = ("F1", "The two errors cost about the same — optimise the balanced score.")
        return (f"Classification — optimise {headline[0]}", headline[1])

    return ("Unknown decision type", "Set DECISION_TYPE to 'severity' or 'routing'.")


frame, guidance = recommend_evaluation(
    DECISION_TYPE, COST_OF_MISSED_CASE, COST_OF_FALSE_ALARM, HAS_HISTORICAL_LABEL
)

print("=" * 68)
print(f"Decision      : {MY_DECISION}")
print(f"Framed as     : {DECISION_TYPE}")
print("-" * 68)
print(f"Evaluation    : {frame}")
print(f"Why           : {guidance}")
print("=" * 68)
print("\nRule of thumb: name the DECISION first — it picks the metric. "
      "\nNever pick a metric because it looks good.")


> ✓ **CHECKPOINT — Section 7**
> You've taken the workshop's core evaluation move and applied it to your own use case: **decision → frame → headline metric.** That's the skill that transfers, long after you forget this notebook's specific numbers.


---

## Section 8 — From Workshop to Production

📘 **KNOWLEDGE POINT — A benchmark result is a starting point, not a verdict**
You now have evidence for whether SAP-RPT-1 is good enough — but "good enough" is a moving target once the model meets real, drifting data.


## Summary

Exercise 1B extends the same JIT use case with an evaluation and benchmarking lens:
- **Regression** helps estimate delay severity and potential business impact
- **Classification** helps route intervention decisions faster
- **SAP-RPT-1 versus open-source baselines** gives a practical comparison point for architecture and delivery choices

Use the results in this notebook to discuss what is strong enough for advisory rollout, what business decisions each modeling approach supports best, and what additional evidence would be needed before broader operationalization.

---

> ### 🏗️ From Workshop to Production
>
> This benchmark is **workshop-grade**. Before you trust a model in production, hold onto these questions:
> - Which metric becomes your production SLA — MAE, recall on Red-risk POs, or a business-value-weighted score?
> - How do you keep the holdout honest and re-validate as supplier behavior shifts over time?
> - When does a benchmark result justify retraining — or switching models?
>
> *→ After the hands-on, we'll work through these in an instructor-led architecture deep dive — the **Architecture Playbook** — on moving from workshop-grade to production.*
